In [ ]:
!apt install aria2

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libaria2-0 libc-ares2
The following NEW packages will be installed:
  aria2 libaria2-0 libc-ares2
0 upgraded, 3 newly installed, 0 to remove and 2 not upgraded.
Need to get 1,513 kB of archives.
After this operation, 5,441 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libc-ares2 amd64 1.18.1-1ubuntu0.22.04.3 [45.1 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libaria2-0 amd64 1.36.0-1 [1,086 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 aria2 amd64 1.36.0-1 [381 kB]
Fetched 1,513 kB in 1s (1,935 kB/s)
Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
Unpacking libc-ares2:amd64 (1.18.1-1ubunt

# PMC_VQA

In [ ]:
import os
import csv
import zipfile
import re
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from collections import Counter

def clean_answer(answer_raw: str, choice_map: dict) -> str:
    """
    Resolve answer from raw answer string:
    - If it's a single letter A/B/C/D → map to choice text
    - If it matches 'A: some text' or 'A. some text' → strip the prefix
    - If choice text itself starts with 'X: ' or 'X. ' → strip the prefix
    - If it's already clean text → return as-is
    - If empty/nan/none → return ''
    """
    if not answer_raw:
        return ''

    answer_raw = answer_raw.strip()

    # Treat literal 'nan' / 'none' as empty
    if answer_raw.lower() in ('nan', 'none', 'n/a', ''):
        return ''

    # Case 1: bare single letter  →  'B'
    if re.fullmatch(r'[A-D]', answer_raw, re.IGNORECASE):
        label = answer_raw.upper()
        raw_choice = choice_map.get(label, '')
        return strip_choice_prefix(raw_choice)

    # Case 2: 'B: some text' or 'B. some text' or 'B) some text'
    m = re.match(r'^([A-D])[:\.\)]\s*(.+)$', answer_raw, re.IGNORECASE)
    if m:
        label      = m.group(1).upper()
        inline_text = m.group(2).strip()
        # prefer the inline text (already has the answer), but cross-check
        # with the choice map if available
        raw_choice = choice_map.get(label, '')
        choice_text = strip_choice_prefix(raw_choice)
        # return whichever is longer / more informative
        return choice_text if choice_text else inline_text

    # Case 3: plain free text — just strip any leading choice prefix
    return strip_choice_prefix(answer_raw)


def strip_choice_prefix(text: str) -> str:
    """Remove leading 'A: ', 'A. ', 'A) ' prefix from a choice string."""
    if not text:
        return ''
    m = re.match(r'^[A-D][:\.\)]\s*(.+)$', text.strip(), re.IGNORECASE)
    return m.group(1).strip() if m else text.strip()


def convert_pmc_vqa_to_csv(
    dataset_name,
    target_dir,
    output_dir,
    splits=['train', 'validation', 'test'],
    image_dir='./dataset/pmc_images'
):
    print(f"Loading dataset: {dataset_name}")

    os.makedirs(image_dir, exist_ok=True)
    os.makedirs(target_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)

    # ── Download & unzip image archives (skip if already done) ───────────────
    ZIP_FILES = ['images.zip', 'images_2.zip']
    for zip_name in ZIP_FILES:
        zip_local = os.path.join(target_dir, zip_name)

        if os.path.exists(zip_local):
            print(f"✓ {zip_name} already downloaded, skipping.")
        else:
            print(f"\nDownloading {zip_name}...")
            try:
                zip_local = hf_hub_download(
                    repo_id=dataset_name,
                    filename=zip_name,
                    repo_type='dataset',
                    cache_dir=target_dir,
                    local_dir=target_dir
                )
                print(f"  ✓ Downloaded to {zip_local}")
            except Exception as e:
                print(f"  ✗ Could not download {zip_name}: {e}")
                continue

        marker = os.path.join(image_dir, f".extracted_{zip_name}")
        if os.path.exists(marker):
            print(f"✓ {zip_name} already extracted, skipping.")
            continue

        print(f"Extracting {zip_name} → {image_dir} ...")
        try:
            with zipfile.ZipFile(zip_local, 'r') as zf:
                members = zf.namelist()
                for i, member in enumerate(members):
                    filename = os.path.basename(member)
                    if not filename:
                        continue
                    dest = os.path.join(image_dir, filename)
                    if not os.path.exists(dest):
                        with zf.open(member) as src, open(dest, 'wb') as out:
                            out.write(src.read())
                    if (i + 1) % 5000 == 0:
                        print(f"  Extracted {i + 1}/{len(members)}")
            open(marker, 'w').close()
            print(f"  ✓ Done extracting {zip_name}")
        except Exception as e:
            print(f"  ✗ Error extracting {zip_name}: {e}")

    # ── Build image lookup ────────────────────────────────────────────────────
    print("\nBuilding image lookup index...")
    image_lookup = {}
    for fname in os.listdir(image_dir):
        if fname.startswith('.'):
            continue
        image_lookup[fname] = os.path.join(image_dir, fname)
        image_lookup[os.path.splitext(fname)[0]] = os.path.join(image_dir, fname)
    print(f"  → {len(image_lookup)} images indexed")

    def resolve_image(fig_path: str) -> str:
        if not fig_path:
            return ''
        basename = os.path.basename(fig_path)
        if basename in image_lookup:
            return image_lookup[basename]
        if os.path.exists(fig_path):
            return fig_path
        candidate = os.path.join(image_dir, fig_path)
        if os.path.exists(candidate):
            return candidate
        parts = fig_path.replace('\\', '/').split('/')
        for i in range(1, len(parts)):
            sub = os.path.join(image_dir, *parts[i:])
            if os.path.exists(sub):
                return sub
        return ''

    SPLIT_FILES = {
        'train': [
            ('hf://datasets/RadGenome/PMC-VQA/train.csv',   'freetext'),
            ('hf://datasets/RadGenome/PMC-VQA/train_2.csv', 'choice'),
        ],
        'validation': [
            ('hf://datasets/RadGenome/PMC-VQA/valid.csv', 'choice'),
        ],
        'test': [
            ('hf://datasets/RadGenome/PMC-VQA/test.csv',   'freetext'),
            ('hf://datasets/RadGenome/PMC-VQA/test_2.csv', 'choice'),
        ],
    }

    split_rows     = {s: [] for s in splits}
    failed_rows    = []
    seen_questions = set()
    seen_captions  = set()
    successful, duplicates = 0, 0

    for split in splits:
        if split not in SPLIT_FILES:
            print(f"Warning: Unknown split '{split}', skipping.")
            continue

        for csv_file, answer_type in SPLIT_FILES[split]:
            file_tag = os.path.splitext(os.path.basename(csv_file))[0]
            try:
                print(f"\nLoading: {csv_file}  [answer_type={answer_type}]")
                ds = load_dataset('csv', data_files=csv_file, cache_dir=target_dir, split='train')
                print(f"  → {len(ds)} rows, columns: {ds.column_names}")

                for idx, sample in enumerate(ds):
                    try:
                        question = (sample.get('Question')    or '').strip()
                        caption  = (sample.get('Caption')     or '').strip()
                        fig_path = (sample.get('Figure_path') or '').strip()

                        choice_map = {
                            'A': (sample.get('Choice A') or '').strip(),
                            'B': (sample.get('Choice B') or '').strip(),
                            'C': (sample.get('Choice C') or '').strip(),
                            'D': (sample.get('Choice D') or '').strip(),
                        }

                        def log_fail(reason):
                            failed_rows.append({
                                "source_file":  csv_file,
                                "row_index":    idx,
                                "split":        split,
                                "answer_type":  answer_type,
                                "Figure_path":  fig_path,
                                "Question":     question,
                                "Caption":      caption,
                                "Answer_raw":   (sample.get('Answer')       or '').strip(),
                                "Answer_label": (sample.get('Answer_label') or '').strip(),
                                "Choice_A":     choice_map['A'],
                                "Choice_B":     choice_map['B'],
                                "Choice_C":     choice_map['C'],
                                "Choice_D":     choice_map['D'],
                                "reason":       reason,
                            })

                        if not question and not caption:
                            log_fail("missing_question_and_caption")
                            continue

                        # ── Resolve answer ────────────────────────────────
                        if answer_type == 'freetext':
                            # freetext Answer_raw may still contain 'B: some text'
                            answer_raw = (sample.get('Answer') or '').strip()
                            answer = clean_answer(answer_raw, choice_map)
                        else:
                            # choice: prefer Answer_label, fallback to Answer_raw
                            label      = (sample.get('Answer_label') or '').strip().upper()
                            answer_raw = (sample.get('Answer')       or '').strip()
                            if label in ('A', 'B', 'C', 'D'):
                                answer = strip_choice_prefix(choice_map.get(label, ''))
                            else:
                                # label missing or invalid — try Answer_raw
                                answer = clean_answer(answer_raw, choice_map)

                        # ── Resolve image path ────────────────────────────
                        image_save_path = resolve_image(fig_path)

                        # ── Caption row ───────────────────────────────────
                        if caption and caption not in seen_captions:
                            seen_captions.add(caption)
                            split_rows[split].append({
                                "image_path":  image_save_path,
                                "caption":     caption,
                                "is_caption":  True,
                                "question":    "",
                                "answer":      "",
                                "data_source": dataset_name,
                                "split":       split,
                            })
                            successful += 1

                        # ── QA row ────────────────────────────────────────
                        if question:
                            if not answer:
                                log_fail("missing_answer")
                            elif question in seen_questions:
                                duplicates += 1
                            else:
                                seen_questions.add(question)
                                split_rows[split].append({
                                    "image_path":  image_save_path,
                                    "caption":     "",
                                    "is_caption":  False,
                                    "question":    question,
                                    "answer":      answer,
                                    "data_source": dataset_name,
                                    "split":       split,
                                })
                                successful += 1

                        if (idx + 1) % 500 == 0:
                            print(f"  [{file_tag}] Processed {idx + 1}/{len(ds)}")

                    except Exception as e:
                        log_fail(f"exception: {e}")

            except Exception as e:
                print(f"Error loading {csv_file}: {e}")
                continue

    # ── Save split CSVs ───────────────────────────────────────────────────────
    fieldnames = ["image_path", "caption", "is_caption", "question", "answer", "data_source", "split"]
    output_files = {}
    for split, rows in split_rows.items():
        if not rows:
            continue
        out_path = os.path.join(output_dir, f"pmc_vqa_{split}.csv")
        with open(out_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        output_files[split] = out_path

    # ── Save failed CSV ───────────────────────────────────────────────────────
    failed_csv_path = os.path.join(output_dir, "pmc_vqa_failed.csv")
    if failed_rows:
        fail_fields = [
            "source_file", "row_index", "split", "answer_type",
            "Figure_path", "Question", "Caption",
            "Answer_raw", "Answer_label",
            "Choice_A", "Choice_B", "Choice_C", "Choice_D",
            "reason"
        ]
        with open(failed_csv_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fail_fields)
            writer.writeheader()
            writer.writerows(failed_rows)

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'='*50}")
    print(f"✓ Written     {successful} rows")
    print(f"✗ Failed      {len(failed_rows)}  → pmc_vqa_failed.csv")
    print(f"⊘ Duplicates  {duplicates} skipped")

    reason_counts = Counter(r['reason'] for r in failed_rows)
    print(f"\nFailure reasons:")
    for reason, count in reason_counts.most_common():
        print(f"  {reason}: {count}")

    print(f"\nOutput CSVs:")
    for split, path in output_files.items():
        qa_count  = sum(1 for r in split_rows[split] if not r['is_caption'])
        cap_count = sum(1 for r in split_rows[split] if r['is_caption'])
        print(f"  {split}: {qa_count} QA  |  {cap_count} captions  →  {path}")

    return output_files, failed_csv_path

In [ ]:
pmc_vqa_dataset = convert_pmc_vqa_to_csv(
    dataset_name="RadGenome/PMC-VQA",
    target_dir="./output/pmc_vqa",
    output_dir="./output/pmc_vqa",
    image_dir="./output/pmc_vqa/images",
    splits=['train', 'test'],
)

Loading dataset: RadGenome/PMC-VQA



images.zip:   0%|          | 0.00/18.9G [00:00<?, ?B/s]

  ✓ Downloaded to output/pmc_vqa/images.zip
Extracting images.zip → ./output/pmc_vqa/images ...
  Extracted 5000/149076
  Extracted 10000/149076
  Extracted 15000/149076
  Extracted 20000/149076
  Extracted 25000/149076
  Extracted 30000/149076
  Extracted 35000/149076
  Extracted 40000/149076
  Extracted 45000/149076
  Extracted 50000/149076
  Extracted 55000/149076
  Extracted 60000/149076
  Extracted 65000/149076
  Extracted 70000/149076
  Extracted 75000/149076
  Extracted 80000/149076
  Extracted 85000/149076
  Extracted 90000/149076
  Extracted 95000/149076
  Extracted 100000/149076
  Extracted 105000/149076
  Extracted 110000/149076
  Extracted 115000/149076
  Extracted 120000/149076
  Extracted 125000/149076
  Extracted 130000/149076
  Extracted 135000/149076
  Extracted 140000/149076
  Extracted 145000/149076
  ✓ Done extracting images.zip



images_2.zip:   0%|          | 0.00/2.21G [00:00<?, ?B/s]

  ✓ Downloaded to output/pmc_vqa/images_2.zip
Extracting images_2.zip → ./output/pmc_vqa/images ...
  Extracted 5000/164361
  Extracted 10000/164361
  Extracted 15000/164361
  Extracted 20000/164361
  Extracted 25000/164361
  Extracted 30000/164361
  Extracted 35000/164361
  Extracted 40000/164361
  Extracted 45000/164361
  Extracted 50000/164361
  Extracted 55000/164361
  Extracted 60000/164361
  Extracted 65000/164361
  Extracted 70000/164361
  Extracted 75000/164361
  Extracted 80000/164361
  Extracted 85000/164361
  Extracted 90000/164361
  Extracted 95000/164361
  Extracted 100000/164361
  Extracted 105000/164361
  Extracted 110000/164361
  Extracted 115000/164361
  Extracted 120000/164361
  Extracted 125000/164361
  Extracted 130000/164361
  Extracted 135000/164361
  Extracted 140000/164361
  Extracted 145000/164361
  Extracted 150000/164361
  Extracted 155000/164361
  Extracted 160000/164361
  ✓ Done extracting images_2.zip

Building image lookup index...
  → 626870 images index

...

Generating train split: 0 examples [00:00, ? examples/s]

  → 50000 rows, columns: ['Figure_path', 'Question', 'Answer', 'Choice A', 'Choice B', 'Choice C', 'Choice D', 'Answer_label']
  [test] Processed 500/50000
  [test] Processed 1000/50000
  [test] Processed 1500/50000
  [test] Processed 2000/50000
  [test] Processed 2500/50000
  [test] Processed 3000/50000
  [test] Processed 3500/50000
  [test] Processed 4000/50000
  [test] Processed 4500/50000
  [test] Processed 5000/50000
  [test] Processed 5500/50000
  [test] Processed 6000/50000
  [test] Processed 6500/50000
  [test] Processed 7000/50000
  [test] Processed 7500/50000
  [test] Processed 8000/50000
  [test] Processed 8500/50000
  [test] Processed 9000/50000
  [test] Processed 9500/50000
  [test] Processed 10000/50000
  [test] Processed 10500/50000
  [test] Processed 11000/50000
  [test] Processed 11500/50000
  [test] Processed 12000/50000
  [test] Processed 12500/50000
  [test] Processed 13000/50000
  [test] Processed 13500/50000
  [test] Processed 14000/50000
  [test] Processed 14500/

test_2.csv:   0%|          | 0.00/12.4M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

  → 33430 rows, columns: ['index', 'Figure_path', 'Caption', 'Question', 'Choice A', 'Choice B', 'Choice C', 'Choice D', 'Answer', 'split']
  [test_2] Processed 500/33430
  [test_2] Processed 1000/33430
  [test_2] Processed 1500/33430
  [test_2] Processed 2000/33430
  [test_2] Processed 2500/33430
  [test_2] Processed 3000/33430
  [test_2] Processed 3500/33430
  [test_2] Processed 4000/33430
  [test_2] Processed 4500/33430
  [test_2] Processed 5000/33430
  [test_2] Processed 5500/33430
  [test_2] Processed 6000/33430
  [test_2] Processed 6500/33430
  [test_2] Processed 7000/33430
  [test_2] Processed 7500/33430
  [test_2] Processed 8000/33430
  [test_2] Processed 8500/33430
  [test_2] Processed 9000/33430
  [test_2] Processed 9500/33430
  [test_2] Processed 10000/33430
  [test_2] Processed 10500/33430
  [test_2] Processed 11000/33430
  [test_2] Processed 11500/33430
  [test_2] Processed 12000/33430
  [test_2] Processed 12500/33430
  [test_2] Processed 13000/33430
  [test_2] Processed 1

In [ ]:
import pandas as pd

df = pd.read_csv("/content/output/pmc_vqa/pmc_vqa_failed.csv")
df

,source_file,row_index,split,answer_type,Figure_path,Question,Caption,Answer_raw,Answer_label,Choice_A,Choice_B,Choice_C,Choice_D,reason
0,hf://datasets/RadGenome/PMC-VQA/train.csv,1706,train,freetext,PMC1976316_F1.jpg,What is the result of the CT scan?,NaN,NaN,A,A: N/A,B: Large filling defect in the left atrium,C: Blocked coronary arteries,D: None of the above.,missing_answer
1,hf://datasets/RadGenome/PMC-VQA/train.csv,2758,train,freetext,PMC2394263_fig1.jpg,What does the MRI result demonstrate for the p...,NaN,NaN,D,A:Grade II glioma progressing to grade III,B:Grade II glioma transforming to GBM,C:GBM reducing to Grade II glioma,D:N/A,missing_answer
2,hf://datasets/RadGenome/PMC-VQA/train.csv,7496,train,freetext,PMC2796441_fig2.jpg,How many inferior vena cavas are visible in th...,NaN,NaN,D,A: 1,B: 2,C: 3,D: None,missing_answer
3,hf://datasets/RadGenome/PMC-VQA/train.csv,11175,train,freetext,PMC2945635_fig2.jpg,What type of contrast is used in this CT scan?,NaN,none,D,A: Gastrografin,B: Iodine-based,C: Barium,D: none,missing_answer
4,hf://datasets/RadGenome/PMC-VQA/train.csv,13879,train,freetext,PMC3081313_pone-0018944-g007.jpg,Which column in the right panels of (A) and (B...,NaN,NaN,C,A: Projection,B: Average,C: None,D: Both,missing_answer
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,hf://datasets/RadGenome/PMC-VQA/test.csv,23979,test,freetext,PMC8776059_FIG1.jpg,What is the feature represented by the left up...,NaN,NaN,D,A:Parenchymal opacities,B:Tree/budding opacities,C:Bronchiectasis,D:N/A,missing_answer
81,hf://datasets/RadGenome/PMC-VQA/test.csv,31648,test,freetext,PMC8958406_figure2.jpg,What are the atherosclerotic changes seen in F...,NaN,NaN,B,A: Surrounding the lesion,B: None,C: Near the superior mesenteric artery,D: Inside the superior mesenteric artery,missing_answer
82,hf://datasets/RadGenome/PMC-VQA/test.csv,33869,test,freetext,PMC9013826_F2.jpg,How many MitraClips are shown in the image?,NaN,NaN,D,A: One,B: Two,C: Three,D: None,missing_answer
83,hf://datasets/RadGenome/PMC-VQA/test.csv,33963,test,freetext,PMC9017908_F1.jpg,What additional radiograph was taken besides t...,NaN,NaN,D,A: Lateral view of both shoulders,B: Oblique view of both shoulders,C: Posterior view of both shoulders,D: None,missing_answer


In [ ]:
!rm -rf /content/output/pmc_vqa/images.zip
!rm -rf /content/output/pmc_vqa/images_2.zip

# ROCO 2

In [ ]:
import os
import csv
import zipfile
import subprocess
from pathlib import Path
from collections import Counter

def download_file(url: str, dest_dir: str, filename: str = None) -> str | None:
    """Download with aria2c, skip if already exists."""
    Path(dest_dir).mkdir(parents=True, exist_ok=True)
    if filename is None:
        filename = url.split('/')[-1].split('?')[0]
    dest_path = os.path.join(dest_dir, filename)

    if os.path.exists(dest_path):
        print(f"✓ {filename} already exists, skipping.")
        return dest_path

    print(f"Downloading {filename}...", end=' ', flush=True)
    try:
        subprocess.run([
            'aria2c',
            '--console-log-level=error',
            '-c', '-x', '16', '-s', '16', '-k', '1M',
            '--summary-interval=0', '--quiet',
            '-d', dest_dir,
            '-o', filename,
            url
        ], check=True, capture_output=True, text=True)
        print("Done!")
        return dest_path
    except subprocess.CalledProcessError as e:
        print(f"\n✗ Failed: {e.stderr.strip()}")
        return None


def convert_rocov2_to_csv(
    output_dir:  str,
    work_dir:    str = './dataset/rocov2',
    image_dir:   str = './dataset/rocov2/images',
):
    DATASET_NAME = 'zenodo/rocov2'
    BASE_URL     = 'https://zenodo.org/records/10821435/files'

    SPLIT_FILES = {
        'train':      'train_captions.csv',
        'validation': 'valid_captions.csv',
        'test':       'test_captions.csv',
    }
    IMAGE_ZIPS = {
        'train':      'train_images.zip',
        'validation': 'valid_images.zip',
        'test':       'test_images.zip',
    }

    os.makedirs(work_dir,   exist_ok=True)
    os.makedirs(image_dir,  exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)

    # ── Download caption CSVs ─────────────────────────────────────────────────
    print("\n── Downloading caption CSVs ──────────────────────────────")
    for split, fname in SPLIT_FILES.items():
        download_file(f"{BASE_URL}/{fname}", work_dir, fname)

    # ── Download, extract image zips (skip if already done) ──────────────────
    print("\n── Downloading & extracting image zips ───────────────────")
    for split, zip_name in IMAGE_ZIPS.items():
        marker = os.path.join(image_dir, f".extracted_{zip_name}")
        zip_path = os.path.join(work_dir, zip_name)

        # Download
        if not os.path.exists(zip_path):
            download_file(f"{BASE_URL}/{zip_name}", work_dir, zip_name)
        else:
            print(f"✓ {zip_name} already downloaded, skipping.")

        # Extract
        if os.path.exists(marker):
            print(f"✓ {zip_name} already extracted, skipping.")
            continue

        if not os.path.exists(zip_path):
            print(f"✗ {zip_name} not found, skipping extraction.")
            continue

        print(f"Extracting {zip_name} → {image_dir} ...")
        try:
            with zipfile.ZipFile(zip_path, 'r') as zf:
                members = zf.namelist()
                for i, member in enumerate(members):
                    filename = os.path.basename(member)
                    if not filename:
                        continue
                    dest = os.path.join(image_dir, filename)
                    if not os.path.exists(dest):
                        with zf.open(member) as src, open(dest, 'wb') as out:
                            out.write(src.read())
                    if (i + 1) % 5000 == 0:
                        print(f"  Extracted {i + 1}/{len(members)}")
            open(marker, 'w').close()
            print(f"  ✓ Done extracting {zip_name}")
        except Exception as e:
            print(f"  ✗ Error extracting {zip_name}: {e}")

    # ── Build image lookup ────────────────────────────────────────────────────
    print("\nBuilding image lookup index...")
    image_lookup = {}
    for fname in os.listdir(image_dir):
        if fname.startswith('.'):
            continue
        image_lookup[fname] = os.path.join(image_dir, fname)
        image_lookup[os.path.splitext(fname)[0]] = os.path.join(image_dir, fname)
    print(f"  → {len(image_lookup)} images indexed")

    def resolve_image(img_name: str) -> str:
        if not img_name:
            return ''
        basename = os.path.basename(img_name)
        if basename in image_lookup:
            return image_lookup[basename]
        if os.path.exists(img_name):
            return img_name
        candidate = os.path.join(image_dir, img_name)
        if os.path.exists(candidate):
            return candidate
        # strip leading dirs one level at a time
        parts = img_name.replace('\\', '/').split('/')
        for i in range(1, len(parts)):
            sub = os.path.join(image_dir, *parts[i:])
            if os.path.exists(sub):
                return sub
        return ''

    # ── Process caption CSVs ──────────────────────────────────────────────────
    print("\n── Processing caption CSVs ───────────────────────────────")
    split_rows   = {s: [] for s in SPLIT_FILES}
    failed_rows  = []
    seen_captions = set()
    successful, duplicates = 0, 0

    for split, csv_fname in SPLIT_FILES.items():
        csv_path = os.path.join(work_dir, csv_fname)
        if not os.path.exists(csv_path):
            print(f"✗ {csv_fname} not found, skipping.")
            continue

        print(f"\nProcessing {csv_fname} ...")
        with open(csv_path, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            print(f"  → columns: {reader.fieldnames}")

            for idx, row in enumerate(reader):
                try:
                    # RocoV2 columns: ID, caption  (image file = ID + .jpg)
                    img_id  = (row.get('ID') or row.get('id') or '').strip()
                    caption = (row.get('caption') or row.get('Caption') or '').strip()

                    if not caption:
                        failed_rows.append({
                            "source_file": csv_fname,
                            "row_index":   idx,
                            "split":       split,
                            "ID":          img_id,
                            "caption":     caption,
                            "reason":      "missing_caption",
                        })
                        continue

                    if caption in seen_captions:
                        duplicates += 1
                        continue
                    seen_captions.add(caption)

                    # Image filename is ID.jpg
                    image_save_path = resolve_image(f"{img_id}.jpg") if img_id else ''

                    split_rows[split].append({
                        "image_path":  image_save_path,
                        "caption":     caption,
                        "is_caption":  True,
                        "question":    "",
                        "answer":      "",
                        "data_source": DATASET_NAME,
                        "split":       split,
                    })
                    successful += 1

                    if (idx + 1) % 1000 == 0:
                        print(f"  Processed {idx + 1} rows")

                except Exception as e:
                    failed_rows.append({
                        "source_file": csv_fname,
                        "row_index":   idx,
                        "split":       split,
                        "ID":          '',
                        "caption":     '',
                        "reason":      f"exception: {e}",
                    })

    # ── Save split CSVs ───────────────────────────────────────────────────────
    fieldnames = ["image_path", "caption", "is_caption", "question", "answer", "data_source", "split"]
    output_files = {}
    for split, rows in split_rows.items():
        if not rows:
            continue
        out_path = os.path.join(output_dir, f"rocov2_{split}.csv")
        with open(out_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        output_files[split] = out_path

    # ── Save failed CSV ───────────────────────────────────────────────────────
    if failed_rows:
        failed_path = os.path.join(output_dir, "rocov2_failed.csv")
        with open(failed_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=["source_file", "row_index", "split", "ID", "caption", "reason"])
            writer.writeheader()
            writer.writerows(failed_rows)
        print(f"\n  ✗ Failed rows → {failed_path}")

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'='*50}")
    print(f"✓ Written     {successful} rows")
    print(f"✗ Failed      {len(failed_rows)}")
    print(f"⊘ Duplicates  {duplicates} skipped")

    if failed_rows:
        print(f"\nFailure reasons:")
        for reason, count in Counter(r['reason'] for r in failed_rows).most_common():
            print(f"  {reason}: {count}")

    print(f"\nOutput CSVs:")
    for split, path in output_files.items():
        print(f"  {split}: {len(split_rows[split])} captions  →  {path}")

    return output_files

In [ ]:
convert_rocov2_to_csv(
    output_dir="./output/roco_v2",
    work_dir='./output/roco_v2',
    image_dir='./output/roco_v2/images')


── Downloading caption CSVs ──────────────────────────────

── Downloading & extracting image zips ───────────────────
Extracting train_images.zip → ./output/roco_v2/images ...
  Extracted 5000/59959
  Extracted 10000/59959
  Extracted 15000/59959
  Extracted 20000/59959
  Extracted 25000/59959
  Extracted 30000/59959
  Extracted 35000/59959
  Extracted 40000/59959
  Extracted 45000/59959
  Extracted 50000/59959
  Extracted 55000/59959
  ✓ Done extracting train_images.zip
Extracting valid_images.zip → ./output/roco_v2/images ...
  Extracted 5000/9905
  ✓ Done extracting valid_images.zip
Extracting test_images.zip → ./output/roco_v2/images ...
  Extracted 5000/9928
  ✓ Done extracting test_images.zip

Building image lookup index...
  → 159578 images indexed

── Processing caption CSVs ───────────────────────────────

Processing train_captions.csv ...
  → columns: ['ID', 'Caption']
  Processed 1000 rows
  Processed 2000 rows
  Processed 3000 rows
  Processed 4000 rows
  Processed 5000 r

{'train': './output/roco_v2/rocov2_train.csv',
 'validation': './output/roco_v2/rocov2_validation.csv',
 'test': './output/roco_v2/rocov2_test.csv'}

In [ ]:
!rm -rf "/content/output/roco_v2/test_images.zip"
!rm -rf "/content/output/roco_v2/train_images.zip"
!rm -rf "/content/output/roco_v2/valid_images.zip"

!rm -rf "/content/output/roco_v2/test_captions.csv"
!rm -rf "/content/output/roco_v2/train_captions.csv"
!rm -rf "/content/output/roco_v2/valid_captions.csv"

# ROCO 1

In [ ]:
# !mkdir -p /root/.kaggle
# !mv /content/kaggle.json /root/.kaggle/
# !chmod 600 /root/.kaggle/kaggle.json

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("virajbagal/roco-dataset")

print("Path to dataset files:", path)

100%|██████████| 6.19G/6.19G [01:46<00:00, 62.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/virajbagal/roco-dataset/versions/1


In [ ]:
import os
import csv
import shutil
from pathlib import Path
from collections import Counter
from PIL import Image


DATASET_NAME = 'kaggle/rocov1'

SPLIT_CONFIG = {
    'train': {
        'csv':    'train/radiologytraindata.csv',
        'images': 'train/radiology/images/',
    },
    'validation': {
        'csv':    'validation/radiologyvaldata.csv',
        'images': 'validation/radiology/images/',
    },
    'test': {
        'csv':    'test/radiologytestdata.csv',
        'images': 'test/radiology/images/',
    },
}


def verify_image(image_path: str) -> bool:
    try:
        with Image.open(image_path) as img:
            img.verify()
        return True
    except Exception:
        return False


def convert_rocov1_to_csv(
    output_dir: str,
    data_dir:   str = '/root/.cache/kagglehub/datasets/virajbagal/roco-dataset/versions/1/all_data',
):
    # ── image_dir lives inside output_dir ─────────────────────────────────────
    image_dir = os.path.join(output_dir, 'images')
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(image_dir,  exist_ok=True)

    if not os.path.exists(data_dir):
        print(f"✗ Data directory not found: {data_dir}")
        return {}

    print(f"✓ Using data from: {data_dir}")
    print(f"✓ Images will be copied to: {image_dir}")

    # ── Process each split ────────────────────────────────────────────────────
    print("\n── Processing splits ─────────────────────────────────────")
    split_rows    = {s: [] for s in SPLIT_CONFIG}
    failed_rows   = []
    seen_captions = set()
    successful, duplicates, copied, skipped_copy = 0, 0, 0, 0

    for split, config in SPLIT_CONFIG.items():
        csv_path    = os.path.join(data_dir, config['csv'])
        images_base = os.path.join(data_dir, config['images'])

        if not os.path.exists(csv_path):
            print(f"✗ CSV not found: {csv_path}, skipping {split}.")
            continue

        if not os.path.exists(images_base):
            print(f"✗ Images dir not found: {images_base}, skipping {split}.")
            continue

        print(f"\nProcessing {split}")
        print(f"  CSV:    {csv_path}")
        print(f"  Images: {images_base}")

        def log_fail(img_name, caption, reason):
            failed_rows.append({
                "source_file": config['csv'],
                "split":       split,
                "name":        img_name,
                "caption":     caption,
                "reason":      reason,
            })

        with open(csv_path, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            print(f"  → columns: {reader.fieldnames}")

            for idx, row in enumerate(reader):
                try:
                    img_name = (row.get('name') or '').strip()
                    caption  = (row.get('caption') or '').strip()

                    if not caption:
                        log_fail(img_name, caption, "missing_caption")
                        continue

                    if caption in seen_captions:
                        duplicates += 1
                        continue

                    # ── Check source image exists ─────────────────────────
                    src_path = os.path.join(images_base, img_name) if img_name else ''
                    if not src_path or not os.path.exists(src_path):
                        log_fail(img_name, caption, "image_not_found")
                        continue

                    if not verify_image(src_path):
                        log_fail(img_name, caption, "image_corrupt")
                        continue

                    # ── Copy image to output images dir ───────────────────
                    dest_path = os.path.join(image_dir, img_name)
                    if not os.path.exists(dest_path):
                        shutil.copy2(src_path, dest_path)
                        copied += 1
                    else:
                        skipped_copy += 1

                    seen_captions.add(caption)
                    split_rows[split].append({
                        "image_path":  dest_path,   # ← points to unified images dir
                        "caption":     caption,
                        "is_caption":  True,
                        "question":    "",
                        "answer":      "",
                        "data_source": DATASET_NAME,
                        "split":       split,
                    })
                    successful += 1

                    if (idx + 1) % 1000 == 0:
                        print(f"  Processed {idx + 1} rows  (copied: {copied}, skipped: {skipped_copy})")

                except Exception as e:
                    log_fail(row.get('name', ''), row.get('caption', ''), f"exception: {e}")

        print(f"  ✓ {split}: {len(split_rows[split])} valid rows")

    # ── Save split CSVs ───────────────────────────────────────────────────────
    fieldnames = ["image_path", "caption", "is_caption", "question", "answer", "data_source", "split"]
    output_files = {}
    for split, rows in split_rows.items():
        if not rows:
            continue
        out_path = os.path.join(output_dir, f"rocov1_{split}.csv")
        with open(out_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        output_files[split] = out_path

    # ── Save failed CSV ───────────────────────────────────────────────────────
    if failed_rows:
        failed_path = os.path.join(output_dir, "rocov1_failed.csv")
        with open(failed_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=["source_file", "split", "name", "caption", "reason"])
            writer.writeheader()
            writer.writerows(failed_rows)
        print(f"\n  ✗ Failed rows → {failed_path}")

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'='*50}")
    print(f"✓ Written     {successful} rows")
    print(f"✓ Copied      {copied} images  |  {skipped_copy} already existed")
    print(f"✗ Failed      {len(failed_rows)}")
    print(f"⊘ Duplicates  {duplicates} skipped")

    if failed_rows:
        print(f"\nFailure reasons:")
        for reason, count in Counter(r['reason'] for r in failed_rows).most_common():
            print(f"  {reason}: {count}")

    print(f"\nOutput structure:")
    print(f"  {output_dir}/")
    print(f"  ├── images/          ({copied + skipped_copy} images)")
    for split, path in output_files.items():
        print(f"  ├── rocov1_{split}.csv  ({len(split_rows[split])} rows)")

    return output_files

In [ ]:
# ── Run ───────────────────────────────────────────────────────────────────────
convert_rocov1_to_csv(
    output_dir='./output/roco_v1',
    data_dir='/root/.cache/kagglehub/datasets/virajbagal/roco-dataset/versions/1/all_data'
)

✓ Using data from: /root/.cache/kagglehub/datasets/virajbagal/roco-dataset/versions/1/all_data
✓ Images will be copied to: ./output/roco_v1/images

── Processing splits ─────────────────────────────────────

Processing train
  CSV:    /root/.cache/kagglehub/datasets/virajbagal/roco-dataset/versions/1/all_data/train/radiologytraindata.csv
  Images: /root/.cache/kagglehub/datasets/virajbagal/roco-dataset/versions/1/all_data/train/radiology/images/
  → columns: ['id', 'name', 'caption']
  Processed 1000 rows  (copied: 1000, skipped: 0)
  Processed 2000 rows  (copied: 2000, skipped: 0)
  Processed 3000 rows  (copied: 2999, skipped: 0)
  Processed 4000 rows  (copied: 3995, skipped: 0)
  Processed 5000 rows  (copied: 4990, skipped: 0)
  Processed 6000 rows  (copied: 5984, skipped: 0)
  Processed 7000 rows  (copied: 6978, skipped: 0)
  Processed 8000 rows  (copied: 7974, skipped: 0)
  Processed 9000 rows  (copied: 8971, skipped: 0)
  Processed 10000 rows  (copied: 9966, skipped: 0)
  Processe

{'train': './output/roco_v1/rocov1_train.csv',
 'validation': './output/roco_v1/rocov1_validation.csv',
 'test': './output/roco_v1/rocov1_test.csv'}

In [ ]:
!rm -rf /kaggle/working/*
!rm -rf /root/.cache/kagglehub/datasets/*

# Path_VQA

In [ ]:
import os
import csv
from datasets import load_dataset
from collections import Counter
from PIL import Image


DATASET_NAME = 'moebouassida/path-vqa-enhanced'


def convert_pathvqa_to_csv(
    output_dir: str,
    cache_dir:  str = './dataset/pathvqa',
    image_dir:  str = './dataset/pathvqa/images',
):
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(cache_dir,  exist_ok=True)
    os.makedirs(image_dir,  exist_ok=True)

    print(f"Loading dataset: {DATASET_NAME}")

    try:
        ds = load_dataset(DATASET_NAME, cache_dir=cache_dir)
        print(f"  → splits found: {list(ds.keys())}")
    except Exception as e:
        print(f"✗ Failed to load dataset: {e}")
        return {}

    SPLIT_MAP = {
        'train':      'train',
        'validation': 'validation',
        'valid':      'validation',
        'test':       'test',
    }

    split_rows     = {}
    failed_rows    = []
    seen_qa        = set()   # deduplicate on (question, answer) pair
    successful, duplicates = 0, 0

    for hf_split, split_data in ds.items():
        split = SPLIT_MAP.get(hf_split, hf_split)
        if split not in split_rows:
            split_rows[split] = []

        print(f"\nProcessing {hf_split} ({len(split_data)} rows) → split='{split}'")
        print(f"  → columns: {split_data.column_names}")

        for idx, sample in enumerate(split_data):
            try:
                question        = (sample.get('question')        or '').strip()
                enhanced_answer = (sample.get('enhanced_answer') or '').strip()
                raw_answer      = (sample.get('answer')          or '').strip()
                pil_image       = sample.get('image')

                # ── Resolve best answer: enhanced first, fallback to raw ───
                answer = enhanced_answer if enhanced_answer else raw_answer

                # ── Validate ──────────────────────────────────────────────
                if not question:
                    failed_rows.append({
                        "split": split, "row_index": idx,
                        "question": question, "answer": answer,
                        "reason": "missing_question"
                    })
                    continue

                if not answer:
                    failed_rows.append({
                        "split": split, "row_index": idx,
                        "question": question, "answer": answer,
                        "reason": "missing_answer"
                    })
                    continue

                # ── Deduplicate on (question, answer) pair ────────────────
                qa_key = (question.lower(), answer.lower())
                if qa_key in seen_qa:
                    duplicates += 1
                    continue

                # ── Save image ────────────────────────────────────────────
                if not isinstance(pil_image, Image.Image):
                    failed_rows.append({
                        "split": split, "row_index": idx,
                        "question": question, "answer": answer,
                        "reason": "missing_image"
                    })
                    continue

                image_filename  = f"pathvqa_{hf_split}_{idx:06d}.jpg"
                image_save_path = os.path.join(image_dir, image_filename)
                if not os.path.exists(image_save_path):
                    pil_image.convert('RGB').save(image_save_path, 'JPEG')

                seen_qa.add(qa_key)
                split_rows[split].append({
                    "image_path":  image_save_path,
                    "caption":     "",
                    "is_caption":  False,
                    "question":    question,
                    "answer":      answer,
                    "data_source": DATASET_NAME,
                    "split":       split,
                })
                successful += 1

                if (idx + 1) % 1000 == 0:
                    print(f"  Processed {idx + 1}/{len(split_data)}")

            except Exception as e:
                failed_rows.append({
                    "split": split, "row_index": idx,
                    "question": "", "answer": "",
                    "reason": f"exception: {e}"
                })

        print(f"  ✓ {split}: {len(split_rows[split])} rows")

    # ── Save split CSVs ───────────────────────────────────────────────────────
    fieldnames = ["image_path", "caption", "is_caption", "question", "answer", "data_source", "split"]
    output_files = {}
    for split, rows in split_rows.items():
        if not rows:
            continue
        out_path = os.path.join(output_dir, f"pathvqa_{split}.csv")
        with open(out_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        output_files[split] = out_path

    # ── Save failed CSV ───────────────────────────────────────────────────────
    if failed_rows:
        failed_path = os.path.join(output_dir, "pathvqa_failed.csv")
        with open(failed_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=["split", "row_index", "question", "answer", "reason"])
            writer.writeheader()
            writer.writerows(failed_rows)
        print(f"\n  ✗ Failed rows → {failed_path}")

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'='*50}")
    print(f"✓ Written     {successful} rows")
    print(f"✗ Failed      {len(failed_rows)}")
    print(f"⊘ Duplicates  {duplicates} skipped")

    if failed_rows:
        print(f"\nFailure reasons:")
        for reason, count in Counter(r['reason'] for r in failed_rows).most_common():
            print(f"  {reason}: {count}")

    print(f"\nOutput CSVs:")
    for split, path in output_files.items():
        print(f"  {split}: {len(split_rows[split])} QA rows  →  {path}")

    return output_files

In [ ]:
# ── Run ───────────────────────────────────────────────────────────────────────
convert_pathvqa_to_csv(
    output_dir='./output/pathvqa',
    cache_dir='./output/pathvqa',
    image_dir='./output/pathvqa/images',
)

Loading dataset: moebouassida/path-vqa-enhanced


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00005.parquet:   0%|          | 0.00/270M [00:00<?, ?B/s]

data/train-00001-of-00005.parquet:   0%|          | 0.00/289M [00:00<?, ?B/s]

data/train-00002-of-00005.parquet:   0%|          | 0.00/344M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/6719 [00:00<?, ? examples/s]

  → splits found: ['train', 'validation', 'test']

Processing train (19654 rows) → split='train'
  → columns: ['image', 'question', 'answer', 'enhanced_answer']
  Processed 1000/19654
  Processed 2000/19654
  Processed 3000/19654
  Processed 4000/19654
  Processed 5000/19654
  Processed 6000/19654
  Processed 7000/19654
  Processed 8000/19654
  Processed 9000/19654
  Processed 10000/19654
  Processed 11000/19654
  Processed 12000/19654
  Processed 13000/19654
  Processed 14000/19654
  Processed 15000/19654
  Processed 16000/19654
  Processed 17000/19654
  Processed 18000/19654
  Processed 19000/19654
  ✓ train: 19360 rows

Processing validation (6259 rows) → split='validation'
  → columns: ['image', 'question', 'answer', 'enhanced_answer']
  Processed 1000/6259
  Processed 2000/6259
  Processed 3000/6259
  Processed 4000/6259
  Processed 5000/6259
  Processed 6000/6259
  ✓ validation: 6060 rows

Processing test (6719 rows) → split='test'
  → columns: ['image', 'question', 'answer', 'en

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


  Processed 6000/6719
  ✓ test: 3647 rows

✓ Written     29067 rows
✗ Failed      0
⊘ Duplicates  3565 skipped

Output CSVs:
  train: 19360 QA rows  →  ./output/pathvqa/pathvqa_train.csv
  validation: 6060 QA rows  →  ./output/pathvqa/pathvqa_validation.csv
  test: 3647 QA rows  →  ./output/pathvqa/pathvqa_test.csv


{'train': './output/pathvqa/pathvqa_train.csv',
 'validation': './output/pathvqa/pathvqa_validation.csv',
 'test': './output/pathvqa/pathvqa_test.csv'}

# VQA_rad

In [ ]:
import os
import csv
from datasets import load_dataset
from collections import Counter
from PIL import Image


DATASET_NAME = 'flaviagiammarino/vqa-rad'


def convert_vqarad_to_csv(
    output_dir: str,
    cache_dir:  str = './dataset/vqarad',
    image_dir:  str = './dataset/vqarad/images',
):
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(cache_dir,  exist_ok=True)
    os.makedirs(image_dir,  exist_ok=True)

    print(f"Loading dataset: {DATASET_NAME}")

    try:
        ds = load_dataset(DATASET_NAME, cache_dir=cache_dir)
        print(f"  → splits found: {list(ds.keys())}")
    except Exception as e:
        print(f"✗ Failed to load dataset: {e}")
        return {}

    SPLIT_MAP = {
        'train':      'train',
        'validation': 'validation',
        'valid':      'validation',
        'test':       'test',
    }

    split_rows     = {}
    failed_rows    = []
    seen_qa        = set()
    successful, duplicates = 0, 0

    for hf_split, split_data in ds.items():
        split = SPLIT_MAP.get(hf_split, hf_split)
        if split not in split_rows:
            split_rows[split] = []

        print(f"\nProcessing {hf_split} ({len(split_data)} rows) → split='{split}'")
        print(f"  → columns: {split_data.column_names}")

        for idx, sample in enumerate(split_data):
            try:
                question  = (sample.get('question') or '').strip()
                answer    = (sample.get('answer')   or '').strip()
                pil_image = sample.get('image')

                if not question:
                    failed_rows.append({"split": split, "row_index": idx,
                                        "question": question, "answer": answer,
                                        "reason": "missing_question"})
                    continue

                if not answer:
                    failed_rows.append({"split": split, "row_index": idx,
                                        "question": question, "answer": answer,
                                        "reason": "missing_answer"})
                    continue

                qa_key = (question.lower(), answer.lower())
                if qa_key in seen_qa:
                    duplicates += 1
                    continue

                if not isinstance(pil_image, Image.Image):
                    failed_rows.append({"split": split, "row_index": idx,
                                        "question": question, "answer": answer,
                                        "reason": "missing_image"})
                    continue

                image_filename  = f"vqarad_{hf_split}_{idx:06d}.jpg"
                image_save_path = os.path.join(image_dir, image_filename)
                if not os.path.exists(image_save_path):
                    pil_image.convert('RGB').save(image_save_path, 'JPEG')

                seen_qa.add(qa_key)
                split_rows[split].append({
                    "image_path":  image_save_path,
                    "caption":     "",
                    "is_caption":  False,
                    "question":    question,
                    "answer":      answer,
                    "data_source": DATASET_NAME,
                    "split":       split,
                })
                successful += 1

                if (idx + 1) % 500 == 0:
                    print(f"  Processed {idx + 1}/{len(split_data)}")

            except Exception as e:
                failed_rows.append({"split": split, "row_index": idx,
                                    "question": "", "answer": "",
                                    "reason": f"exception: {e}"})

        print(f"  ✓ {split}: {len(split_rows[split])} rows")

    # ── Save split CSVs ───────────────────────────────────────────────────────
    fieldnames = ["image_path", "caption", "is_caption", "question", "answer", "data_source", "split"]
    output_files = {}
    for split, rows in split_rows.items():
        if not rows:
            continue
        out_path = os.path.join(output_dir, f"vqarad_{split}.csv")
        with open(out_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        output_files[split] = out_path

    # ── Save failed CSV ───────────────────────────────────────────────────────
    if failed_rows:
        failed_path = os.path.join(output_dir, "vqarad_failed.csv")
        with open(failed_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=["split", "row_index", "question", "answer", "reason"])
            writer.writeheader()
            writer.writerows(failed_rows)
        print(f"\n  ✗ Failed rows → {failed_path}")

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'='*50}")
    print(f"✓ Written     {successful} rows")
    print(f"✗ Failed      {len(failed_rows)}")
    print(f"⊘ Duplicates  {duplicates} skipped")

    if failed_rows:
        print(f"\nFailure reasons:")
        for reason, count in Counter(r['reason'] for r in failed_rows).most_common():
            print(f"  {reason}: {count}")

    print(f"\nOutput CSVs:")
    for split, path in output_files.items():
        print(f"  {split}: {len(split_rows[split])} QA rows  →  {path}")

    return output_files

In [ ]:
# ── Run ───────────────────────────────────────────────────────────────────────
convert_vqarad_to_csv(
    output_dir='./output/vqarad',
    cache_dir='./output/vqarad',
    image_dir='./output/vqarad/images',
)

Loading dataset: flaviagiammarino/vqa-rad


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-eb8844602202be(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

data/test-00000-of-00001-e5bc3d208bb4dee(…):   0%|          | 0.00/10.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1793 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/451 [00:00<?, ? examples/s]

  → splits found: ['train', 'test']

Processing train (1793 rows) → split='train'
  → columns: ['image', 'question', 'answer']
  Processed 500/1793
  Processed 1000/1793
  Processed 1500/1793
  ✓ train: 1691 rows

Processing test (451 rows) → split='test'
  → columns: ['image', 'question', 'answer']
  ✓ test: 395 rows

✓ Written     2086 rows
✗ Failed      0
⊘ Duplicates  158 skipped

Output CSVs:
  train: 1691 QA rows  →  ./output/vqarad/vqarad_train.csv
  test: 395 QA rows  →  ./output/vqarad/vqarad_test.csv


{'train': './output/vqarad/vqarad_train.csv',
 'test': './output/vqarad/vqarad_test.csv'}

# Combine

In [ ]:
!ls /content/output

pathvqa  pmc_vqa  roco_v1  roco_v2  vqarad


In [ ]:
import os
import csv
import shutil
from collections import Counter, defaultdict
from PIL import Image
from datasets import Dataset, DatasetDict, Features, Value, Image as HFImage
import pandas as pd


DATASET_DIRS = {
    'pathvqa': './output/pathvqa',
    'pmc_vqa': './output/pmc_vqa',
    'roco_v1': './output/roco_v1',
    'roco_v2': './output/roco_v2',
    'vqarad':  './output/vqarad',
}

SPLIT_NAMES = ['train', 'validation', 'test']


def find_csvs(dataset_dir: str) -> dict:
    found = {}
    for fname in os.listdir(dataset_dir):
        if not fname.endswith('.csv') or 'failed' in fname:
            continue
        for split in SPLIT_NAMES:
            if split in fname:
                found[split] = os.path.join(dataset_dir, fname)
    return found


def verify_image(path: str) -> bool:
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


def resolve_image_path(image_path: str, combined_images: str) -> str:
    if not image_path:
        return ''
    if os.path.exists(image_path):
        return image_path
    filename  = os.path.basename(image_path)
    name, ext = os.path.splitext(filename)
    dest = os.path.join(combined_images, filename)
    if os.path.exists(dest):
        return dest
    parent        = os.path.basename(os.path.dirname(os.path.dirname(image_path)))
    dest_disambig = os.path.join(combined_images, f"{parent}_{name}{ext}")
    if os.path.exists(dest_disambig):
        return dest_disambig
    return ''


def move_image(src: str, dest_dir: str, unique_id: str = '') -> str:
    """
    Move image to dest_dir.
    Every row gets its own file — if the src is already in dest_dir or a name
    collision exists, a copy is made with unique_id suffix so no two rows
    share a file path unless they are provably the same bytes.
    """
    filename  = os.path.basename(src)
    name, ext = os.path.splitext(filename)
    dest      = os.path.join(dest_dir, filename)

    abs_src      = os.path.abspath(src)
    abs_dest_dir = os.path.abspath(dest_dir)

    # ── src is already inside combined dir (rerun / previous move) ───────────
    if abs_src.startswith(abs_dest_dir):
        if unique_id:
            new_dest = os.path.join(dest_dir, f"{name}_{unique_id}{ext}")
            if not os.path.exists(new_dest):
                shutil.copy2(src, new_dest)
            return new_dest
        return src

    # ── no collision at dest — just move ─────────────────────────────────────
    if not os.path.exists(dest):
        shutil.move(src, dest)
        return dest

    # ── collision: check if same physical file ────────────────────────────────
    try:
        same_file = os.path.samefile(src, dest)
    except FileNotFoundError:
        same_file = False

    # Either way (same or different file), give this row its own copy
    new_dest = os.path.join(dest_dir, f"{name}_{unique_id}{ext}")
    if not os.path.exists(new_dest):
        if same_file:
            shutil.copy2(dest, new_dest)   # copy from already-moved dest
        else:
            shutil.move(src, new_dest)     # move the different file
    return new_dest


def combine_datasets(
    combined_dir:    str = './output/combined',
    combined_images: str = './output/combined/images',
    hf_repo_id:      str = None,
    hf_token:        str = None,
):
    os.makedirs(combined_dir,    exist_ok=True)
    os.makedirs(combined_images, exist_ok=True)

    stats = {
        'before':          defaultdict(int),
        'after':           defaultdict(int),
        'failed':          [],
        'duplicates':      0,
        'missing_image':   0,
        'corrupt_image':   0,
        'missing_content': 0,
        'moved_images':    0,
        'copied_images':   0,   # copies made for shared/colliding filenames
    }

    seen_qa       = set()
    seen_captions = set()
    split_rows    = {s: [] for s in SPLIT_NAMES}
    global_idx    = 0           # unique id per accepted row

    for ds_name, ds_dir in DATASET_DIRS.items():
        if not os.path.exists(ds_dir):
            print(f"⚠ Skipping {ds_name}: directory not found ({ds_dir})")
            continue

        csvs = find_csvs(ds_dir)
        if not csvs:
            print(f"⚠ Skipping {ds_name}: no CSVs found in {ds_dir}")
            continue

        print(f"\n── {ds_name} {'─' * (44 - len(ds_name))}")

        for split, csv_path in csvs.items():
            print(f"  Loading {split}: {csv_path}")

            with open(csv_path, 'r', encoding='utf-8') as f:
                rows = list(csv.DictReader(f))

            stats['before'][f"{ds_name}/{split}"] = len(rows)
            print(f"  → {len(rows)} rows")

            for row in rows:
                is_caption  = str(row.get('is_caption', '')).strip().lower() in ('true', '1')
                image_path  = (row.get('image_path')  or '').strip()
                caption     = (row.get('caption')     or '').strip()
                question    = (row.get('question')    or '').strip()
                answer      = (row.get('answer')      or '').strip()
                data_source = (row.get('data_source') or ds_name).strip()

                def log_fail(reason):
                    stats['failed'].append({
                        "dataset":    ds_name,
                        "split":      split,
                        "is_caption": is_caption,
                        "image_path": image_path,
                        "caption":    caption,
                        "question":   question,
                        "answer":     answer,
                        "reason":     reason,
                    })

                # ── Validate content ──────────────────────────────────────
                if is_caption:
                    if not caption:
                        stats['missing_content'] += 1
                        log_fail("missing_caption")
                        continue
                else:
                    if not question or not answer:
                        stats['missing_content'] += 1
                        log_fail("missing_question_or_answer")
                        continue

                # ── Resolve image ─────────────────────────────────────────
                resolved_path = resolve_image_path(image_path, combined_images)
                if not resolved_path:
                    stats['missing_image'] += 1
                    log_fail("image_not_found")
                    continue

                if not verify_image(resolved_path):
                    stats['corrupt_image'] += 1
                    log_fail("image_corrupt")
                    continue

                # ── Deduplication on content ──────────────────────────────
                if is_caption:
                    dedup_key = caption.lower().strip()
                    if dedup_key in seen_captions:
                        stats['duplicates'] += 1
                        continue
                    seen_captions.add(dedup_key)
                else:
                    dedup_key = (question.lower().strip(), answer.lower().strip())
                    if dedup_key in seen_qa:
                        stats['duplicates'] += 1
                        continue
                    seen_qa.add(dedup_key)

                # ── Move / copy image — every row owns its file ───────────
                unique_id      = f"{ds_name}_{split}_{global_idx:07d}"
                before_dest    = os.path.abspath(resolved_path).startswith(
                                     os.path.abspath(combined_images))
                new_image_path = move_image(resolved_path, combined_images, unique_id)

                if before_dest:
                    stats['copied_images'] += 1   # was already in dest, got a copy
                else:
                    stats['moved_images']  += 1   # freshly moved from source

                global_idx += 1

                split_rows[split].append({
                    "image_path":  new_image_path,
                    "caption":     caption,
                    "is_caption":  is_caption,
                    "question":    question,
                    "answer":      answer,
                    "data_source": data_source,
                    "split":       split,
                })

    # ── Save combined CSVs ────────────────────────────────────────────────────
    print(f"\n── Saving combined CSVs {'─' * 27}")
    fieldnames   = ["image_path", "caption", "is_caption", "question", "answer", "data_source", "split"]
    output_files = {}
    for split, rows in split_rows.items():
        if not rows:
            continue
        out_path = os.path.join(combined_dir, f"combined_{split}.csv")
        with open(out_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        output_files[split] = out_path
        stats['after'][split] = len(rows)
        print(f"  ✓ {split}: {len(rows)} rows → {out_path}")

    # ── Save failed CSV ───────────────────────────────────────────────────────
    if stats['failed']:
        failed_path = os.path.join(combined_dir, "combined_failed.csv")
        with open(failed_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=[
                "dataset", "split", "is_caption",
                "image_path", "caption", "question", "answer", "reason"
            ])
            writer.writeheader()
            writer.writerows(stats['failed'])
        print(f"  ✗ Failed rows → {failed_path}")

    # ── Summary ───────────────────────────────────────────────────────────────
    total_before = sum(stats['before'].values())
    total_after  = sum(stats['after'].values())

    print(f"\n{'='*50}")
    print(f"BEFORE (per dataset/split):")
    for key, count in sorted(stats['before'].items()):
        print(f"  {key}: {count}")

    print(f"\nAFTER (combined splits):")
    for split in SPLIT_NAMES:
        if split not in stats['after']:
            continue
        rows      = split_rows[split]
        qa_count  = sum(1 for r in rows if not r['is_caption'])
        cap_count = sum(1 for r in rows if r['is_caption'])
        print(f"  {split}: {stats['after'][split]} total  ({qa_count} QA  |  {cap_count} captions)")

    print(f"\n  Total before     : {total_before}")
    print(f"  Total after      : {total_after}")
    print(f"  Removed          : {total_before - total_after}")
    print(f"\n  ⊘ Duplicates     : {stats['duplicates']}")
    print(f"  ✗ Missing image  : {stats['missing_image']}")
    print(f"  ✗ Corrupt image  : {stats['corrupt_image']}")
    print(f"  ✗ Missing content: {stats['missing_content']}")
    print(f"\n  📁 Images moved  : {stats['moved_images']}  (fresh from source)")
    print(f"  📁 Images copied : {stats['copied_images']}  (collision or shared filename — safe copy made)")

    if stats['failed']:
        print(f"\n  Failure reasons:")
        for reason, count in Counter(r['reason'] for r in stats['failed']).most_common():
            print(f"    {reason}: {count}")

    if hf_repo_id:
        push_to_huggingface(split_rows, hf_repo_id, hf_token)

    return output_files, stats


def push_to_huggingface(split_rows: dict, repo_id: str, token: str = None):
    print(f"\n── Pushing to HuggingFace: {repo_id} {'─' * 15}")

    features = Features({
        "image":       HFImage(),
        "caption":     Value("string"),
        "is_caption":  Value("bool"),
        "question":    Value("string"),
        "answer":      Value("string"),
        "data_source": Value("string"),
        "split":       Value("string"),
    })

    hf_splits = {}
    for split, rows in split_rows.items():
        if not rows:
            continue

        print(f"  Building {split} ({len(rows)} rows)...")
        df = pd.DataFrame([{
            "image":       row["image_path"],
            "caption":     row["caption"],
            "is_caption":  row["is_caption"],
            "question":    row["question"],
            "answer":      row["answer"],
            "data_source": row["data_source"],
            "split":       row["split"],
        } for row in rows])

        df = df.sample(frac=1, random_state=2).reset_index(drop=True)
        print(f"  ✓ Shuffled with random_state=2")

        hf_splits[split] = Dataset.from_pandas(df, features=features)

    DatasetDict(hf_splits).push_to_hub(
        repo_id,
        token=token,
        private=False,
    )
    print(f"  ✓ Done → https://huggingface.co/datasets/{repo_id}")

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [ ]:
# ── Run ───────────────────────────────────────────────────────────────────────
combine_datasets(
    combined_dir    = './output/combined',
    combined_images = './output/combined/images',
    hf_repo_id      = 'MohamedAhmedAE/medical-vqa-5-datasets',
)


── pathvqa ─────────────────────────────────────
  Loading validation: ./output/pathvqa/pathvqa_validation.csv
  → 6060 rows
  Loading test: ./output/pathvqa/pathvqa_test.csv
  → 3647 rows
  Loading train: ./output/pathvqa/pathvqa_train.csv
  → 19360 rows

── pmc_vqa ─────────────────────────────────────
  Loading test: ./output/pmc_vqa/pmc_vqa_test.csv
  → 83605 rows
  Loading train: ./output/pmc_vqa/pmc_vqa_train.csv
  → 359396 rows

── roco_v1 ─────────────────────────────────────
  Loading validation: ./output/roco_v1/rocov1_validation.csv
  → 8012 rows
  Loading train: ./output/roco_v1/rocov1_train.csv
  → 64737 rows
  Loading test: ./output/roco_v1/rocov1_test.csv
  → 7991 rows

── roco_v2 ─────────────────────────────────────
  Loading validation: ./output/roco_v2/rocov2_validation.csv
  → 9853 rows
  Loading train: ./output/roco_v2/rocov2_train.csv
  → 59636 rows
  Loading test: ./output/roco_v2/rocov2_test.csv
  → 9869 rows

── vqarad ──────────────────────────────────────
  

Uploading the dataset shards:   0%|          | 0/60 [00:00<?, ? shards/s]

Map:   0%|          | 0/7863 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/79 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   8%|7         | 39.4MB /  508MB            

  ✓ Done → https://huggingface.co/datasets/MohamedAhmedAE/medical-vqa-5-datasets


({'train': './output/combined/combined_train.csv',
  'validation': './output/combined/combined_validation.csv',
  'test': './output/combined/combined_test.csv'},
 {'before': defaultdict(int,
              {'pathvqa/validation': 6060,
               'pathvqa/test': 3647,
               'pathvqa/train': 19360,
               'pmc_vqa/test': 83605,
               'pmc_vqa/train': 359396,
               'roco_v1/validation': 8012,
               'roco_v1/train': 64737,
               'roco_v1/test': 7991,
               'roco_v2/validation': 9853,
               'roco_v2/train': 59636,
               'roco_v2/test': 9869,
               'vqarad/train': 1691,
               'vqarad/test': 395}),
  'after': defaultdict(int,
              {'train': 471741, 'validation': 23903, 'test': 105481}),
  'failed': [],
  'duplicates': 33127,
  'missing_image': 0,
  'corrupt_image': 0,
  'missing_content': 0,
  'moved_images': 441669,
  'copied_images': 159456})